In [1]:
## Installation required
## In qbraid we need to run the cell everytime we open a new session

!pip install qiskit[visualization]
# Use the following if you are on MacOS/zsh
#!pip install 'qiskit[visualization]'==1.1.0
!pip install qiskit_ibm_runtime
!pip install qiskit_aer
!pip install matplotlib
!pip install pylatexenc
!pip install prototype-zne

  Using cached matplotlib-3.11.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached pydot-4.0.1-py3-none-any.whl.metadata (11 kB)
  Using cached pillow-12.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached pylatexenc-2.11-py2.py3-none-any.whl.metadata (3.9 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.4.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.66.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (129 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached pandas-3.0.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached matplotlib-3.

In [2]:
import qiskit
qiskit.__version__

'2.5.2'

In [3]:
## Import packages

#from math import pi
#import numpy as np
import time
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
#from qiskit.circuit import Gate
from qiskit_aer import AerSimulator, QasmSimulator
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.visualization import plot_histogram

#qr = QuantumRegister(size) #128
size = 4*4*8 #nb*h*k = 128
V = QuantumRegister(size) #128
X = QuantumRegister(size) #128
qr = QuantumRegister(288)
t = QuantumRegister(3)
cr = ClassicalRegister(2*size+288+3) #128*2+288+3=547
qc = QuantumCircuit(V, X, qr, t, cr)

In [4]:
num_qubits = qc.num_qubits
print("Number of qubits:", num_qubits)

Number of qubits: 547


In [5]:
key = '000000010010001101000101011001111000'+'0000'+'1101010111100110111101111000100100011010'
#key = '00000001001000110100010101100111100010011010101111001101111011110001001000110100' 
iv = '00000001001000110100010101100111100010011010101111001101111011110001001000110100'  
for i in range(80): #For loading Key
    if (key[i] == "1"):
        qc.x(qr[i]) 
for i in range(80): #For loading IV
    if (iv[i] == "1"):
        qc.x(qr[i+93]) 

qc.x(qr[285])
qc.x(qr[286])
qc.x(qr[287])

def key_gen2(qc, qr, V, round, ks):
    for i in range(round):        
        qc.cx(qr[65],qr[92])
        qc.ccx(qr[90],qr[91],qr[92])
        qc.cx(qr[170],qr[92])
        qc.swap(qr[92],t[0])
        
        qc.cx(qr[161],qr[176])
        qc.ccx(qr[174],qr[175],qr[176])
        qc.cx(qr[263],qr[176])
        qc.swap(qr[176],t[1])
        
        qc.cx(qr[242],qr[287])
        qc.ccx(qr[285],qr[286],qr[287])
        qc.cx(qr[68],qr[287])
        qc.swap(qr[287],t[2])

        for j in range(287,176,-1):
            if j == 177:
                qc.swap(qr[177], t[1])
            else:
                qc.swap(qr[j], qr[j-1])

        for j in range(176,92,-1):
            if j == 93:
                qc.swap(qr[93], t[0])
            else:
                qc.swap(qr[j], qr[j-1])
        
        for j in range(92,-1,-1):
            if j == 0:
                qc.swap(qr[0], t[2])
            else:
                qc.swap(qr[j], qr[j-1])


    """ Key Stream Generation """
    for i in range(ks):
        
        qc.cx(qr[65],qr[92])
        qc.swap(qr[92],t[0])
        
        qc.cx(qr[161],qr[176])
        qc.swap(qr[176],t[1])
        
        qc.cx(qr[242],qr[287])
        qc.swap(qr[287],t[2])
        
        qc.cx(t[0],V[i])
        qc.cx(t[1],V[i])
        qc.cx(t[2],V[i])
        
        qc.ccx(qr[90],qr[91],t[0]) 
        qc.cx(qr[170],t[0]) 
        
        qc.ccx(qr[174],qr[175],t[1]) 
        qc.cx(qr[263],t[1]) 
        
        qc.ccx(qr[285],qr[286],t[2]) 
        qc.cx(qr[68],t[2]) 
        
        for j in range(287,176,-1):
            if j == 177:
                qc.swap(qr[177], t[1])
            else:
                qc.swap(qr[j], qr[j-1])

        for j in range(176,92,-1):
            if j == 93:
                qc.swap(qr[93], t[0])
            else:
                qc.swap(qr[j], qr[j-1])
        
        for j in range(92,-1,-1):
            if j == 0:
                qc.swap(qr[0], t[2])
            else:
                qc.swap(qr[j], qr[j-1])
                


In [ ]:
def algo1_org(qc, V, X):
    nb = 4; delta = 2; H = 4; W = 8
    #pi1 = ??; pi2 = ??; pi3 = ??; pi4 = ??
    pi1=[2 for i in range(H)]; pi2=[2 for i in range(H)]  #some dummy values
    for j in range(1, nb+1):
        if j%delta == 0:
            algo2_org(j, pi1, pi2, H, W)
        for h in range(1, H+1):
            for k in range(1, W+1):
                index_k = (h - 1) * W + k
                index1 = (j - 1) * H * W + (h - 1) * W + k
                index2 = (j - 1) * H * W + (pi1[h-1] - 1) * W + k
                qc.swap(V[index1-1], X[index_k-1])
                qc.cx(V[index2-1], X[index_k-1])
                index3 = (j - 1) * H * W + (pi2[h-1] - 1) * W + k
                qc.cx(V[index3-1], X[index_k-1])
        
        for h in range(1, H+1):
            for k in range(1, W+1):
                index_k = (h - 1) * W + k
                index_k1 = (h - 1) * W + (k - 2) % W #in place of 12
                qc.cx(X[index_k1-1], X[index_k-1])
                index_k2 = (h - 1) * W + (k + 5) % W #in place of 25
                index_k3 = (h - 1) * W + (k - 7) % W #in place of 27
                qc.cx(X[index_k2-1], X[index_k-1])
                qc.cx(X[index_k3-1], X[index_k-1])
                qc.swap(V[((j-1) * H * W + index_k)-1], X[index_k-1])
                
        for h in range(1, H+1):
            for k in range(1, W+1):
                index_x = (h - 1) * W + k
                index1 = (j - 1) * H * W + (h-1) * W + k
                index2 = (j - 1) * H * W + ((h - pi1[h-1]) % W) + k #in place of (j - 1) * H * W + ((h - pi1[h-1]) % W - 1) * W + k
                qc.swap(V[index2-1], X[index_x-1])
                qc.swap(V[index1-1], X[index_x-1])
                qc.swap(V[index2-1], X[index_x-1])
                

def algo2_org(j, pi1, pi2, H, W):
    for h in range(1, H+1):
        for k in range(1, W+1):
            index_x = (h - 1) * W + k
            index1 = (j - 1) * H * W + (h-1) * W + k
            index2 = (j - 1) * H * W + (pi1[h-1] - 1) * W + k
            index3 = (j - 1) * H * W + (pi2[h-1] - 1) * W + k
            qc.ccx(V[index1-1], X[index2-1], X[index_x-1])
            qc.cx(V[index3-1], X[index_x-1])
            qc.swap(V[index1-1], X[index_x-1])

In [7]:
start = time.perf_counter()
key_gen2(qc, qr, V, 1152, 128) # This is a part of initialization
algo1_org(qc, V, X)

In [8]:
qc.measure_all()
print("Circuit depth: ", qc.depth())

Circuit depth:  126469


In [9]:
# Simulate the quantum circuit
backend = QasmSimulator(method='matrix_product_state') # 'automatic' or 'matrix_product_state', 'density_matrix' (but only 'matrix_product_state' is working for select = 7; for other options simulation fails)
sampler = Sampler(backend)
job = sampler.run([qc], shots=1)
result = job.result()[0]
print("Time elapsed (in sec):", time.perf_counter() - start)

Time elapsed (in sec): 19.41358151500026


In [10]:
# Plot results
#counts = result.data.creg.get_counts() 
#if qc.measure_all() is used, write result.data.meas.get_counts()
counts = result.data.meas.get_counts()
#print(counts)
for idx, (k,val) in enumerate(counts.items()):
    op_rev = k[::-1]
print(op_rev)
#print("V (initial):", V_inp)
print("Keystream:", op_rev[0:128])

0011111110111011000011100100111001110111100101110111010110100001110000010110001011010111101001100101100110010110011110001110010100100001001010001100010111001010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000100001101110001010100001011100010000000010001000100110000000110010000110110111000101111001111010001010111011001011111100011001111000011000000010000010001000101010101000110000101010011010110001110101011001111101101101011111100100000101101000000101001010100111110101000000110110111011101011000
Keystream: 00111111101110110000111001001110011101111001011101110101101000011100000101100010110101111010011001011001100101100111100011100101
